我目前的训练数据包括粗粒度图结构数据和对应的核磁数据，这些数据取自有机小分子，最大碳数量不超过80。粗粒度分子是对全原子分子的粗粒度转化，具体包括33个粗粒度结构单元su，其中26个是含碳结构单元，且这些结构单元只含有一个碳原子，其余7个为不含碳结构单元，每个结构单元除H外其余原子均是固定的，部分结构单元的H原子是变化的（根据其连接的周围结构单元数量决定）；粗粒度的图结构数据包括节点特征矩阵x（总计39维数据，0-32为one-hot su编码，33-38为结构单元的元素组成CHONSX（X为卤组元素）），邻接矩阵edge_index和边特征矩阵edge_attr（2维数据，0-1,分别表示键级，是否在环上两个特征），全局元素组成total_atom_counts（CHONSX 6维数据），全局结构单元分布su_hist（33维结构单元one-hot）；核磁数据是0-300ppm，总计3001个数据点的向量数据，使用了洛伦兹函数对核磁的峰进行扩展，生成固定半峰宽为1ppm的洛伦兹峰，注意核磁数据为稀疏数据，大部分位置没有峰，强度为0，核磁数据经过峰强度归一化，最大峰强度为1；此外，还有含碳节点对应的节点级核磁信息，包括每个含碳节点对应的峰中心和强度。

首先应该直接根据得到S2N预测得到结构单元分布H_target直接对每个su分配1-hop类型（所有的su，包括非碳su，全部分配1-hop，这些1-hop应该来自子图数据库中每类结构单元su存在的1-hop组成类型），然后查看这些模版的1-hop是否满足H_target的约束，每个中心su（假设su是甲烷CH3-，只能外接一个结构单元su，那么1-hop就需要满足所有的中心su的su_max_degree（1）x H_target（CH3-））个该类su），所有的su的1-hop组成分布结果需要硬性满足H_target。

然后，为已经确定好的中心su+1-hop从子图数据库选取存在2-hop类型，及针对每个su分配完整的子图模版，注意这里的2-hop同样会受到H_target的约束（假设su是-CH2-，能外接两个结构单元su，假设这两个su分别是-CH2-和-O-，那么该中心su-CH2-就能够在连接两个2-hop节点，对应的2-hop只允许存在两个该类su，就是说2-hop满足的约束为每个su连接的1-hop的su类型的su_max_degree之和减去1-hop节点数量），并分配每个模版对应的z（每个模版对应的核磁峰居中的那个特征z，由子图数据库中进行过统计的），然后计算含碳节点的核磁，并与目标核磁进行比较，然后调整模版,就是调整2-hop，使其能够向目标核磁谱图靠近。

最后再细选每个模版库存在的z，根据已经重构出来的核磁与目标核磁在强度缺口处的差异，选取更加合适的z.  这样分为多层来实现模版的选取和最终z的选取。这里强调1-hop的强约束满足H_target，2-hop可以适当放宽H_target的约束。主要的难点是怎么调整H_target使得最终的H_target以及子图模版，z及其最终预测的核磁结果能够高度匹配目标核磁。


1-hop和2-hop需要满足H_target约束的意思是：假如存在节点A（度数为2）：1个，B节点（度数为2）：2个，C节点（度数为3）：1个，组成子图B-A-B-C；对于1-hop，由于A的度数为2，那么单个A节点能够出现在1-hop两次，当以B节点为中心节点时，子图中1-hop可以出现两个A节点（A的中心节点只有1个，但是由于A的度数为2，因此A可以在1-hop出现2次）；对于2-hop，由于A连接的1-hop节点是两个B节点，且B节点的度数为2，因此A在2-hop中出现的次数就等于1-hop节点的度数之和减去1-hop的节点总数，及两个B节点（度数为2）能否连接4个其余节点，减去1-hop节点本身与中心节点的连接数2，因此子图中2-hop能够出现A节点的次数也是2个，及中心节点在其余子图中以2-hop节点出现的次数为该中心节点的1-hop节点的度数之和减去该中心节点的度数。

### SMILES的粗粒度化

In [ ]:
# Import necessary libraries
import os
import sys
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, SVG
import matplotlib.pyplot as plt

#smiles = "CC1=CC(=CC(=C1)C(=O)N(C(C)(C)C)NC(=O)C2=C(C3=C(C=C2)OCCC3)C)C  " 
#smiles = "C1CN(CCC1CCCC2CCN(CC2)CCO)CCO" 
#smiles = "CSCC[C@@H](C(=O)O)NC(=O)OCC1C2=CC=CC=C2C3=CC=CC=C13" 
#smiles = "CC1=CC(=C(C=C1SC2=CC(=C(C=C2C)O)C(C)(C)C)C(C)(C)C)O" 
#smiles = "C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C=C3)O)O)O" 
#smiles = "C1=CC=C(C=C1)OCC2=CC=CC3=C2C(=CC=C3)COC4=CC=CC=C4"
#smiles = "C1C2=CC=CC=C2C3=CC4=CC=CC=C4C=C31"
#smiles = "COC1=CC=CC=C1N2CCN(CC2)CC(COC3=CC=CC4=CC=CC=C43)O"
#smiles = "CCCCCCCCCC1CCCCC1"
#smiles = "CC(=O)OC1=C(C=C(C=C1)CC=C)OC"
#smiles = "CCCCCCC1=C2C=CC=CC2=CC3=CC=CC=C31"
#smiles = "C1=C2C(=CC3=C1C(=O)NC3=O)C(=O)NC2=O"
#smiles = "C1=CC=C2C=C3C(=O)C=CC(=O)C3=CC2=C1"
#smiles = "CC1=CC(=O)CC(C1)(C)C"
#smiles = "CC1=CNC(=O)NC1=O"
#smiles = "CC1=C(N=CN1)CO"
#smiles = "CCOC1=CC=C(C=C1)C(C)(C)COCC2=CC(=CC=C2)OC3=CC=CC=C3"
#smiles = "CC(C)(C)OC(=O)N1CCC(CC1)N1CCN(CC1)c1nsc(n1)-c1cccs1"
smiles = "C[C@@H](C1=CC2=C(C=C1)C=C(C=C2)OC)C(=O)O"    #19020
name = "PHPA_ester"

# Display the original molecule
mol = Chem.MolFromSmiles(smiles)
img = Draw.MolToImage(mol, size=(400, 300))
plt.figure(figsize=(5, 4))
plt.imshow(img)
plt.title(name)
plt.axis('off')
plt.show()

In [ ]:
!python model/coarse_graph.py --smiles "C1=CC(=CC=C1C[C@@H](C(=O)O)NC(=O)CN)O" --name 4335 --output_dir ./demo

### 数据集生成

In [ ]:
#数据集的生成
!python model/build_dataset.py --data_dir /mnt/e/code/spectra-scraper/NMR_spectral_data/organized_data \
    --shift_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/shift_rdkit_data \
    --output_dir /mnt/e/code/NMRer-result/data3-1 --num_workers 16

In [ ]:
#数据集文件pt的读取
import torch
from model.coarse_graph import visualize_coarse_grained_graph 

graph_path = '/mnt/e/code/NMRer-result/data3-R/38.pt' 
graph_data = torch.load(graph_path,weights_only=True)
print(graph_data)

In [ ]:
!python model/s2n_predict.py \
    --checkpoint checkpoints_s2n/best_s2n_model.pt \
    --spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv \
    --elements "C=243,H=200,O=34,N=1,S=1,X=0" \
    --n_max 1000

       # --spectrum_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/51.csv \

### 数据分析及其数据增强

In [ ]:
#结果输出
!python model/dataset_utils.py analyze \
    --data_dir /mnt/e/code/NMRer-result/data3-R \
    --output_dir analysis_report

In [ ]:
# 可视化节点分布和数量分布
from pathlib import Path
from model.dataset_utils import visualize_analysis_results

visualize_analysis_results(
    Path("analysis_report/su_distribution.csv"),
    Path("analysis_report/node_count_distribution.csv"),
    Path("analysis_report"),
)

In [ ]:
!python model/dataset_utils.py real_stats \
    --data_dir /mnt/e/code/NMRer-result/data3-R \
    --output_dir analysis_report

In [ ]:
!python model/dataset_utils.py split_ppm --data_dir /mnt/e/code/NMRer-result/data3-1 \
    --ok_dir /mnt/e/code/NMRer-result/data3-R \
    --bad_dir /mnt/e/code/NMRer-result/data3-B \
    --margin 0.0 --intensity_threshold 1e-6

In [ ]:
!python model/dataset_utils.py batch_merge \
    --data_dir /mnt/e/code/NMRer-result/data3-R/ \
    --output_dir /mnt/e/code/NMRer-result/data3_R_merged \
    --target_node_counts 220 240 250 260 275 280 300 325 350 400 450 500 550 375 475 425 225 275 375 50 60 70 80 90 100 120 140 160 180 200 \
    --num_per_target 100 \
    #--upsample_sus Alkyl_Cq Aryl_Substituted_Aro_C Carboxylic_Acid Aromatic_Bridgehead_C O_Substituted_Aro_C Alkyl_CH Ether_O Hydroxyl_O Alcohol_Ether_C\
    --upsample_factor 3.0

    # 220 240 260 280 300 350 400 450 500 30 40 50 60 70 80 90 100 120 140 160 180 200 

In [ ]:
# 核磁强度统计
!python model/dataset_utils.py intensity_stats \
    --data_dir /mnt/e/code/NMRer-result/data3-R/ \
    --output_dir analysis_report \
    --threshold 2.0

### G2S子图-->节点级核磁

In [ ]:
!python model/g2s_model.py /mnt/e/code/NMRer-result/data3-R \ 
    --hid 384 \
    --latent_dim 16 \
    --k_hop 3 \
    --beta 0.01 \
    --epochs 10 \
    --batch_size 32 \
    --lr 1e-4
# python g2s_model.py ../data3-R --hid 384 --latent_dim 16 --k_hop 3 --beta 0.01 --epochs 150 --batch_size 24 --lr 1e-5

In [ ]:
!python model/g2s_predict.py --input_path /mnt/e/code/NMRer-result/data1/21797.pt --model_path checkpoints_g2s/g2s_best_model.pt

In [ ]:
!python model/g2s_predict.py \
    --model_path checkpoints_g2s/g2s_best_model.pt \
    --input_path /mnt/e/code/NMRer-result/data3-R/23823.pt \
    # 23823 10915 5013 4335


### S2N 谱图/元素-->节点分布

In [ ]:
# 模型训练
!python model/s2n_model.py \
    /mnt/e/code/NMRer-result/data3-R/ \
    /mnt/e/code/NMRer-result/data3_R_merged \
    --epochs 10 \
    --batch_size 24 \
    --lr 0.00005 

# python model/s2n_model.py ../data3-R/ ../data3_R_merged --epochs 400 --batch_size 24 --lr 0.00005 

In [ ]:
# 验证
!python model/s2n_predict.py \
    --model_path checkpoints_s2n/best_s2n_model.pt \
    --input_path /mnt/e/code/NMRer-result/data3_R_merged/400_09.pt \
    #--num_samples 5

In [ ]:
# 预测
!python model/s2n_predict.py \
    --model_path checkpoints_s2n/best_s2n_model.pt \
    --spectrum_csv standardized_nmr.csv  \
    --elements "C=230, H=200, O=30, N=1, S=1, X=0" \
    --output_name test

       # --spectrum_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/21797.csv \--spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv

### SMILES数据库

In [ ]:
!python model/smiles.py --inputs /mnt/e/code/NMRer-result/chembl_35_chemreps.txt --outdir /mnt/e/code/NMRer-result/smiles/35

In [ ]:
%cd /mnt/e/code/NMRer-result/smiles/35
!bash -lc 'LC_ALL=C find . -maxdepth 1 -type f -name "*.txt" -printf "%P\n" | sort | split -d -l 50000 - /tmp/split_; i=0; for lst in /tmp/split_*; do dir=$(printf "shard_%03d" "$i"); mkdir -p "$dir"; xargs -a "$lst" -I{} mv -- "{}" "$dir"/; i=$((i+1)); done; rm -f /tmp/split_*'

In [ ]:
!python model/build_dataset.py --data_dir smiles/shard_001 --output_dir smiles_out/shard_dataset_001 --num_workers 16

### 3  -hop子图库及其环境编码z

In [ ]:
# 子图数据库
!python model/z_library.py --pt_dir /mnt/e/code/NMRer-result/data3-R \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --out z_library_3/subgraph_library.pt

In [ ]:
!python model/z_library.py --pt_dir smiles_out/shard_dataset_000 \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --base_lib z_library/subgraph_library.pt \
    --merge_in_place

In [ ]:
!python model/z_library.py --pt_dir smiles_out/shard_dataset_001 \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --base_lib z_library/subgraph_library.pt \
    --merge_in_place --mols_per_chunk 1500 \
    --num_workers 12 --prefetch_factor 2 --skip_role_index --amp

In [ ]:
# 统计分析
from model.z_library import analyze_template_sensitivity
analyze_template_sensitivity(
    pt_dir='/mnt/e/code/NMRer-result/data3-R',
    g2s_ckpt='checkpoints_g2s/g2s_best_model.pt',
    lib_path='z_library/subgraph_library.pt',
    out_csv='z_library/template_sensitivity.csv',
    g_samples=64, z_samples=64, device='cuda'
)

### 逆向推理    

In [ ]:
!python model/inverse_pipeline.py \
    --s2n_ckpt checkpoints_s2n/best_s2n_model.pt \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv \
    --lib_path z_library/subgraph_library.pt \
    --su_dist_csv analysis_report/su_distribution.csv \
    --elements "C=230,H=200,O=30,N=1,S=1,X=0" 

In [ ]:
!python /home/dofengqi/Graphnmr/model/inverse_pipeline_v2.py \
  --s2n_ckpt /home/dofengqi/Graphnmr/checkpoints_s2n/best_s2n_model.pt \
  --g2s_ckpt /home/dofengqi/Graphnmr/checkpoints_g2s/g2s_best_model.pt \
  --lib_path /home/dofengqi/Graphnmr/z_library/subgraph_library.pt \
  --spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv \
  --elements "C=230,H=200,O=30,N=1,S=1,X=0" 

### 估计 SU 直方图

In [ ]:
import torch
from pathlib import Path
from model.inverse_pipeline_v2 import InversePipelineV2, _read_spectrum_csv, _parse_elements

spectrum_csv = Path("/mnt/e/NN/GA/NT/standardized_nmr.csv")
elements_str = "C=560,H=436,O=46,N=10,S=4,X=0"

S_target_raw = _read_spectrum_csv(spectrum_csv)
E_target = _parse_elements(elements_str)

if E_target[0] > 0:
    target_area = float(torch.pi) * float(E_target[0].item())
    current_area = float(S_target_raw.sum().item()) * 0.1
    S_target = S_target_raw * (target_area / current_area) if current_area > 1e-6 else S_target_raw
else:
    S_target = S_target_raw

pipeline = InversePipelineV2(
    s2n_ckpt_path=Path("checkpoints_s2n/best_s2n_model.pt"),
    g2s_ckpt_path=Path("checkpoints_g2s/g2s_best_model.pt"),
    lib_path=Path("z_library/subgraph_library.pt"),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)

H_final, logs = pipeline.estimate_su_histogram(S_target, E_target)
print("H_final:", H_final.tolist())


### 1-hop分配

In [ ]:
import torch
from pathlib import Path
from model.inverse_pipeline_v2 import InversePipelineV2, _read_spectrum_csv, _parse_elements

# 1) 读谱与元素
spectrum_csv = Path("/mnt/e/NN/GA/HSQ/standardized_nmr.csv")
elements_str = "C=460,H=400,O=60,N=2,S=2,X=0"

#spectrum_csv = Path("/mnt/e/NN/GA/XLT/standardized_nmr.csv")
#elements_str = "C=300,H=355,O=62,N=7,S=8,X=0"

#spectrum_csv = Path("/mnt/e/NN/GA/LY/standardized_nmr.csv")
#elements_str = "C=600,H=350,O=27,N=7,S=4,X=0"

#spectrum_csv = Path("/mnt/e/NN/GA/MAS/standardized_nmr.csv")
#elements_str = "C=560,H=328,O=20,N=10,S=4,X=0"

#spectrum_csv = Path("/mnt/e/NN/GA/NT/standardized_nmr.csv")
#elements_str = "C=280,H=218,O=23,N=5,S=2,X=0"

S_target_raw = _read_spectrum_csv(spectrum_csv)
E_target = _parse_elements(elements_str)

if E_target[0] > 0:
    target_area = float(torch.pi) * float(E_target[0].item())
    current_area = float(S_target_raw.sum().item()) * 0.1
    S_target = S_target_raw * (target_area / current_area) if current_area > 1e-6 else S_target_raw
else:
    S_target = S_target_raw

# 2) 创建
pipeline = InversePipelineV2(
    s2n_ckpt_path=Path("checkpoints_s2n/best_s2n_model.pt"),
    g2s_ckpt_path=Path("checkpoints_g2s/g2s_best_model.pt"),
    lib_path=Path("z_library_3/subgraph_library.pt"),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)

# 3) 初始化SU直方图
H_final, logs = pipeline.estimate_su_histogram(S_target, E_target)
print("H_final:", H_final.tolist())

# 4) 层1分配
tol_vec = torch.tensor([0.02, 0.1, 0.02, 0.08, 0.08, 0.0], dtype=torch.float)
nodes, H1Use, H1Cap, tokens = pipeline.layer1_assign(
    H_final, E_target, S_target,
    strict_elements=True, tol_vec=tol_vec,
    allow_empty_nodes=False, allow_center_replacement=True,
    h1_overflow_ratio=0.0
)
print(f"Layer1 done. Nodes: {len(nodes)}. See CSV at inverse_result/layer1_assignments.csv")

# 5) 层2分配
H2Use, H2SlotRemain, _, fail_slots, total_slots = pipeline.layer2_assign(
    nodes, H_final, S_target, H1Use, tokens, soft_overflow_ratio=0.18, E_target=E_target
)
print(f"Layer2 done. TotalSlots={total_slots}, FailSlots={fail_slots}.")

# 6) 层3 z-细选
changed = pipeline.layer3_adjust_templates(nodes, S_target, E_target, max_iters=30)
print(f"Layer3 z-tuning done. Changes={changed}.")

# 7) 层4 反向迭代优化
nodes, H4 = pipeline.layer4_iterative_optimize(
    nodes=nodes,
    H_target_int=H_final.clone(),
    S_target=S_target,
    E_target=E_target,
    max_outer_iters=10,
)
print("Layer4 iterative optimization done.")


### 绘图    

In [ ]:
from model.visualize_inverse_results import (
    plot_layer3_su_distribution,
    plot_layer3_local_subgraphs,
)

plot_layer3_su_distribution("inverse_result/layer3_su_distribution.csv",
                            "inverse_result")

plot_layer3_local_subgraphs("inverse_result/layer3_nodes.csv",
                            "inverse_result",
                            num_examples=3)

In [ ]:
# 导入必要的库  
import pandas as pd  
import matplotlib.pyplot as plt  
def plot_ir_data(csv_file):  
    #df = pd.read_csv(csv_file, sep=';', header=None)  
    df = pd.read_csv(csv_file, sep=';', header=None)  
    plt.figure(figsize=(10, 6))  
    plt.plot(df[0], df[1], linestyle='-', color='b') 
    plt.title('NMR Spectrum')  
    plt.xlabel('Wavenumber (cm-1)')  
    plt.ylabel('Intensity (%)')  
    plt.grid()  
    plt.tight_layout()  
    plt.show()  
#csv_file = 'inverse_reconstructed_spectrum.csv'  
#csv_file = '/mnt/e/code/spectra-scraper/NMR_spectral_data/21797.csv'
#csv_file = '/mnt/e/NN/GA/HSQ/standardized_nmr.csv'
csv_file = '/mnt/e/NN/GA/XLT/standardized_nmr.csv'
plot_ir_data(csv_file)

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  

txt_file = '/mnt/e/NN/GA/NMR.txt'
df = pd.read_csv(txt_file, sep='\s+', header=None, engine='python')

plt.figure(figsize=(10, 6))
plt.plot(df[0], df[1], 'b-', linewidth=1.5)
plt.title('NMR Spectrum')
plt.xlabel('Wavenumber (cm⁻¹)')
plt.ylabel('Intensity (%)')
plt.grid(True)
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt  

#graph_path = '/mnt/e/code/NMRer-result/data1/38.pt' 
#graph_path = 'coarse_graphs/83.pt' 
graph_path = '/mnt/e/code/NMRer-result/data2_merged/500_01.pt'
graph_data = torch.load(graph_path,weights_only=True)
#print(graph_data)
y = graph_data['y_spectrum']
plt.figure(figsize=(10, 6))
plt.plot(y, 'b-', linewidth=1.5)
plt.title('NMR Spectrum')
plt.xlabel('Wavenumber (cm⁻¹)')
plt.ylabel('Intensity (%)')
plt.grid(True)
plt.show()